# 🛒 UK Retail Sales Analysis

**Author:** Raveena Somasundaram  
**Dataset:** [Online Retail Dataset – UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/online+retail)  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## 📌 Project Overview

This project analyses transactional data from a UK-based online retailer (2010–2011).  
The goal is to uncover sales trends, top-performing products, customer behaviour, and revenue insights.

### 🔍 Key Questions:
1. What are the monthly and seasonal sales trends?
2. Which products generate the most revenue?
3. Which countries contribute the most to sales?
4. Who are the top customers by spend?
5. What does the customer purchase frequency look like?

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print(' Libraries loaded successfully!')

## 2.  Load the Dataset

> Download the dataset from: https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx  
> Save it as `Online_Retail.xlsx` in the same folder as this notebook.

In [ ]:
df = pd.read_excel('Online_Retail.xlsx', dtype={'CustomerID': str})
print(f'Dataset shape: {df.shape}')
df.head()

## 3.  Data Overview

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

## 4.  Data Cleaning

In [ ]:
print(f'Rows before cleaning: {len(df)}')

# Remove rows with missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove cancelled orders (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove rows with negative or zero Quantity and UnitPrice
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Remove duplicates
df = df.drop_duplicates()

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Parse date and extract Month, Year, Day of Week
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['YearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

print(f'Rows after cleaning: {len(df)}')
print(f'\n✅ Cleaning complete!')
df.head()

## 5. Monthly Revenue Trend

In [ ]:
monthly_revenue = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly_revenue.columns = ['Month', 'Revenue']

plt.figure(figsize=(14, 5))
ax = sns.lineplot(data=monthly_revenue, x='Month', y='Revenue', marker='o', color='steelblue', linewidth=2.5)
plt.title(' Monthly Revenue Trend (2010–2011)', fontsize=15, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.xticks(rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('monthly_revenue.png', dpi=150)
plt.show()
print('\n Insight: Revenue peaks towards Q4, suggesting strong seasonal demand.')

## 6.  Top 10 Best-Selling Products by Revenue

In [ ]:
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=top_products, x='Revenue', y='Description', palette='Blues_d')
plt.title(' Top 10 Products by Revenue', fontsize=15, fontweight='bold')
plt.xlabel('Total Revenue (£)', fontsize=12)
plt.ylabel('Product', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('top_products.png', dpi=150)
plt.show()
print('\n Insight: A small number of products drive a significant portion of total revenue.')

## 7.  Top 10 Countries by Revenue

In [ ]:
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 5))
ax = sns.barplot(data=top_countries, x='Revenue', y='Country', palette='Greens_d')
plt.title(' Top 10 Countries by Revenue', fontsize=15, fontweight='bold')
plt.xlabel('Total Revenue (£)', fontsize=12)
plt.ylabel('Country', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('top_countries.png', dpi=150)
plt.show()
print('\n Insight: The UK dominates revenue, with Netherlands and EIRE as strong secondary markets.')

## 8.  Top 10 Customers by Spend

In [ ]:
top_customers = df.groupby('CustomerID')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 5))
ax = sns.barplot(data=top_customers, x='CustomerID', y='Revenue', palette='Oranges_d')
plt.title(' Top 10 Customers by Total Spend', fontsize=15, fontweight='bold')
plt.xlabel('Customer ID', fontsize=12)
plt.ylabel('Total Spend (£)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('top_customers.png', dpi=150)
plt.show()
print('\n Insight: A small group of high-value customers contribute disproportionately to revenue — a classic Pareto pattern.')

## 9.  Sales by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_revenue = df.groupby('DayOfWeek')['Revenue'].sum().reindex(day_order).reset_index()

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=day_revenue, x='DayOfWeek', y='Revenue', palette='coolwarm')
plt.title(' Revenue by Day of Week', fontsize=15, fontweight='bold')
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('day_of_week.png', dpi=150)
plt.show()
print('\n Insight: Thursday is the busiest day, while weekends show significantly lower activity — typical B2B buying behaviour.')

## 10.  Sales by Hour of Day

In [ ]:
hour_revenue = df.groupby('Hour')['Revenue'].sum().reset_index()

plt.figure(figsize=(12, 5))
ax = sns.lineplot(data=hour_revenue, x='Hour', y='Revenue', marker='o', color='darkorange', linewidth=2.5)
plt.title(' Revenue by Hour of Day', fontsize=15, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig('hour_of_day.png', dpi=150)
plt.show()
print('\n Insight: Peak purchasing hours are between 10am–3pm, suggesting business customers ordering during working hours.')

## 11.  Summary & Key Insights

| # | Insight |
|---|---|
| 1 | Revenue peaks in **Q4 (Oct–Nov)**, indicating strong seasonal demand |
| 2 | A **small number of products** drive the majority of revenue (Pareto principle) |
| 3 | The **United Kingdom** is the dominant market by a large margin |
| 4 | **Top 10 customers** contribute significantly to total revenue — high-value retention is key |
| 5 | **Thursday** is the busiest sales day; weekends are very low — B2B behaviour |
| 6 | **10am–3pm** are peak purchasing hours, aligning with business working hours |

---

##  Next Steps (Future Work)
- RFM Analysis (Recency, Frequency, Monetary) for customer segmentation
- Cohort analysis to track customer retention
- Product recommendation using association rules (Market Basket Analysis)
- Forecasting future revenue using time series models